# Before you train: inspect a MimicGen demonstration
Inspect MimicGen demonstrations **before** training. Use the public example to read action traces and audit results, or bring two files to compare inputs from the same task. The notebook does not train or evaluate a policy.

The default is the [official MimicGen stack source demonstration file](https://huggingface.co/datasets/amandlek/mimicgen_datasets/blob/main/source/stack.hdf5), about 11.8 MB. These are human **source demos**, not generated D0/D1 data. For the optional comparison, supply two existing MimicGen HDF5 files for the same skill, one for each distribution. No format conversion is needed. The notebook does not publish either file; use approved local Jupyter for restricted data.

In [ ]:
# @title Install the pinned tools
%pip install -q --no-warn-conflicts "kalanos[numeric,hdf5]==0.10.1" "huggingface-hub==1.33.0" "plotly==7.1.0"

In [ ]:
# @title Choose the data
from hashlib import file_digest
from pathlib import Path
from huggingface_hub import snapshot_download

MODE = "public_source"  # Optional: "local_pair" for the source HDF5 files you have.
D0_FILE = ""
D1_FILE = ""
OUT = Path("kalanos_mimicgen_outputs")
OUT.mkdir(exist_ok=True)
PUBLIC_REPO = "amandlek/mimicgen_datasets"
REVISION = "89835eddc2abbe2966e43024504b1ff62f914a75"
PUBLIC_FILE = "source/stack.hdf5"
SOURCE_SHA256 = "00331a889de2fa610be9b5c4a0e066f4d387298e0b117181930f166c44076418"

def sha256(path):
    with path.open("rb") as handle:
        return file_digest(handle, "sha256").hexdigest()

if MODE == "public_source":
    root = Path(snapshot_download(repo_id=PUBLIC_REPO, repo_type="dataset",
                                  revision=REVISION, allow_patterns=[PUBLIC_FILE]))
    inputs = {"Public stack source demos": root / PUBLIC_FILE}
    if sha256(inputs["Public stack source demos"]) != SOURCE_SHA256:
        raise ValueError("File hash differs from the pinned publisher pointer.")
elif MODE == "local_pair":
    if not D0_FILE or not D1_FILE:
        raise ValueError("Supply the existing D0 and D1 HDF5 file paths.")
    inputs = {"D0": Path(D0_FILE), "D1": Path(D1_FILE)}
    if any(not p.is_file() for p in inputs.values()):
        raise FileNotFoundError("Both paths must point to files.")
    if sha256(inputs["D0"]) == sha256(inputs["D1"]):
        raise ValueError("D0 and D1 inputs have identical bytes.")
else:
    raise ValueError(f"Unknown input mode: {MODE}")

for name, path in inputs.items():
    print(f"{name}: SHA-256 {sha256(path)}")

In [ ]:
# @title Inventory the HDF5 demo groups and inspect actions
import h5py
import polars as pl
from IPython.display import display

def read_action_inventory(path, plot_demo_limit=4):
    inventory, samples = [], []
    with h5py.File(path, "r") as handle:
        root = handle.get("data")
        if root is None:
            raise ValueError(f"{path.name}: no robomimic /data group.")
        demos = sorted(k for k in root if k.startswith("demo_") and k[5:].isdigit())
        if not demos:
            raise ValueError(f"{path.name}: no demo_N groups found.")
        for key in sorted(demos, key=lambda k: int(k[5:])):
            actions = root[key].get("actions")
            if actions is None or len(actions.shape) != 2 or min(actions.shape) == 0:
                inventory.append({"demo":key,"action_steps":None,"action_dims":None})
                continue
            inventory.append({"demo":key,"action_steps":actions.shape[0],"action_dims":actions.shape[1]})
            if len(samples) < plot_demo_limit:
                # Read just one action channel for plotting, without loading images.
                samples.append((key,actions[:,0]))
    return inventory, samples

inventories, traces = {}, {}
for name, source in inputs.items():
    inventories[name], traces[name] = read_action_inventory(source)
    print(name, ":", len(inventories[name]), "source demos")
    display(pl.DataFrame(inventories[name]).head(8))

In [ ]:
# @title Grade each input once and load the typed report
import shutil
import subprocess
import kalanos

if shutil.which("kalanos") is None:
    raise RuntimeError("Install Kalanos in this runtime first.")
reports = {}
for idx, (name, source) in enumerate(inputs.items()):
    report_path = OUT / f"audit_{idx}.json"
    report_path.unlink(missing_ok=True)  # Never load a leftover report.
    result = subprocess.run(
        ["kalanos", "--verbosity", "error", "grade", str(source),
         "--profile", "numeric-core-v1", "--tier", "standard", "--report", str(report_path)],
        capture_output=True, text=True, check=False, timeout=300)
    print(f"{name}: CLI exit {result.returncode}")
    if not report_path.is_file() or result.returncode not in (0,1):
        raise RuntimeError(result.stderr or result.stdout or "No audit was written.")
    loaded = kalanos.load_report(report_path)
    if not isinstance(loaded, kalanos.Report) or not loaded.episodes:
        raise TypeError("Expected a typed Kalanos report with episode records.")
    reports[name] = loaded

In [ ]:
# @title Compare recorded decisions with a source action view
import plotly.graph_objects as go


def style(fig):
    fig.update_layout(template="simple_white", paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
                      font={"color": "#2b2b29"}, margin={"l": 50, "r": 30, "t": 60, "b": 55})
    fig.update_xaxes(gridcolor="#e1e0d9")
    fig.update_yaxes(gridcolor="#e1e0d9")
    return fig


# The report names a demo `file::data_demo_N`; the `demo` column is the HDF5 group `demo_N`.
rows = [{"input": name, "report_episode": e.id, "demo": e.id.split("::")[-1].removeprefix("data_"),
         "decision": e.eligibility.status.value if e.eligibility else "unknown"}
        for name, report in reports.items() for e in report.episodes]
frame = pl.DataFrame(rows)
display(frame.group_by("input", "decision").len().sort("input", "decision"))
display(frame.head(8))

# Recorded decisions per input; missing measurements are never counted as passes.
decision_colors = {"pass": "#94938d", "review": "#eb6834", "blocked": "#a52e31", "unknown": "#5e6ca5"}
fig = go.Figure()
for decision_name, color in decision_colors.items():
    counts = [frame.filter((pl.col("input") == name) & (pl.col("decision") == decision_name)).height
              for name in inputs]
    fig.add_bar(y=list(inputs), x=counts, orientation="h", name=decision_name.title(),
                marker_color=color, text=[str(n) if n else "" for n in counts])
fig.update_layout(barmode="stack", title="Recorded decisions per input",
                  xaxis_title="Source demos", legend_title_text="Recorded status",
                  legend_traceorder="normal")
style(fig).show()

fig = go.Figure()
for name, items in traces.items():
    for key, values in items:
        fig.add_scatter(x=list(range(len(values))), y=values, name=f"{name} / {key}", mode="lines")
fig.update_layout(title="Source action traces", xaxis_title="Action step within demo",
                  yaxis_title="First action channel (source units)")
style(fig).show()

### Use the evidence
1. Describe one action trace. A step index is not a captured clock, and the trace alone cannot tell whether the task succeeded.
2. Check the local audit's scope. The HDF5 reader infers the episode layout and may synthesize timestamps. Never count missing measurements as passes. Match the `demo` column to the HDF5 demo group before connecting these outputs.
3. If you have matching D0/D1 datasets for the same task, choose a small and a larger training-demo budget at a fixed seed. Record held-out demo IDs and reset settings. Compare performance using the same held-out policy evaluation, not Kalanos status.
4. Write one evidence note: **input hash | demo ID | observation | limitation | keep, inspect further, or recollect**.

In [ ]:
# @title Open the saved audit JSON
from IPython.display import FileLink
for report_path in sorted(OUT.glob("audit_*.json")):
    display(FileLink(str(report_path)))

**Continue with Kalanos:** [Browse dataset reports](https://hub.kalanos.ai) · [Use the grader locally](https://github.com/KalanosAI/kalanos) · [Attach an audit to a training run](https://github.com/KalanosAI/kalanos-tracking)